# Legal Document Summarization Model Training

This notebook demonstrates the training and fine-tuning of summarization models on legal documents.

In [1]:
# Import necessary libraries
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json

# Import our custom modules
from data_loader import LegalDataLoader
from legal_processor import LegalTextProcessor
from extractive_summarizer import LegalExtractiveSummarizer
from abstractive_summarizer import LegalAbstractiveSummarizer
from hybrid_summarizer import LegalHybridSummarizer
from evaluation import LegalSummarizationEvaluator

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'datasets'

## 1. Load and Prepare Dataset

In [ ]:
# Load the dataset
data_loader = LegalDataLoader()
train_data = data_loader.load_in_abs_dataset(split='train')

print(f"Loaded {len(train_data['judgments'])} training samples")

# Create training/validation splits
splits = data_loader.create_training_splits(train_data, train_ratio=0.8, val_ratio=0.1)

print(f"Training samples: {len(splits['train']['judgments'])}")
print(f"Validation samples: {len(splits['validation']['judgments'])}")
print(f"Test samples: {len(splits['test']['judgments'])}")

## 2. Prepare Data for Abstractive Model Training

In [ ]:
# Prepare training data for abstractive model
train_documents = splits['train']['judgments'][:1000]  # Limit for demo
train_summaries = splits['train']['summaries'][:1000]

val_documents = splits['validation']['judgments'][:200]  # Limit for demo
val_summaries = splits['validation']['summaries'][:200]

print(f"Training samples: {len(train_documents)}")
print(f"Validation samples: {len(val_documents)}")

# Create training data in the required format
training_data = [
    {'document': doc, 'summary': summary}
    for doc, summary in zip(train_documents, train_summaries)
]

validation_data = [
    {'document': doc, 'summary': summary}
    for doc, summary in zip(val_documents, val_summaries)
]

## 3. Initialize Abstractive Summarizer

In [ ]:
# Initialize the abstractive summarizer
# Using T5-base for this demo (you can also try 'bart-base' or 'pegasus')
abstractive_summarizer = LegalAbstractiveSummarizer(
    model_name='t5-base',
    device=device
)

print("Abstractive summarizer initialized successfully!")

## 4. Test Pre-trained Model Performance

In [ ]:
# Test the pre-trained model on a few samples
test_samples = 5
test_documents = val_documents[:test_samples]
test_summaries = val_summaries[:test_samples]

print("Testing pre-trained model performance...")

pretrained_results = []
for i, (doc, ref_summary) in enumerate(zip(test_documents, test_summaries)):
    print(f"\nProcessing sample {i+1}/{test_samples}")
    
    # Generate summary
    result = abstractive_summarizer.generate_abstractive_summary(
        doc,
        max_length=200,
        min_length=50
    )
    
    generated_summary = result['summary']
    
    pretrained_results.append({
        'document': doc,
        'reference_summary': ref_summary,
        'generated_summary': generated_summary,
        'metadata': result
    })
    
    print(f"Reference length: {len(ref_summary.split())} words")
    print(f"Generated length: {len(generated_summary.split())} words")
    print(f"Generated summary: {generated_summary[:150]}...")

## 5. Evaluate Pre-trained Model

In [ ]:
# Initialize evaluator
evaluator = LegalSummarizationEvaluator()

# Extract generated and reference summaries
generated_summaries = [result['generated_summary'] for result in pretrained_results]
reference_summaries = [result['reference_summary'] for result in pretrained_results]
original_documents = [result['document'] for result in pretrained_results]

# Evaluate performance
evaluation_results = evaluator.evaluate_summaries(
    generated_summaries=generated_summaries,
    reference_summaries=reference_summaries,
    documents=original_documents
)

print("Pre-trained Model Evaluation Results:")
print("=" * 50)

# Display aggregate scores
aggregate = evaluation_results['aggregate_scores']

if 'rouge' in aggregate:
    rouge_scores = aggregate['rouge']
    print(f"ROUGE-1 F1: {rouge_scores['rouge1_fmeasure_mean']:.4f}")
    print(f"ROUGE-2 F1: {rouge_scores['rouge2_fmeasure_mean']:.4f}")
    print(f"ROUGE-L F1: {rouge_scores['rougeL_fmeasure_mean']:.4f}")

if 'bert_score' in aggregate:
    bert_scores = aggregate['bert_score']
    print(f"BERTScore F1: {bert_scores['f1_mean']:.4f}")

if 'legal_accuracy' in aggregate:
    legal_acc = aggregate['legal_accuracy']
    print(f"Legal Accuracy: {legal_acc['overall_accuracy_mean']:.4f}")

if 'overall' in aggregate:
    overall = aggregate['overall']
    print(f"Overall Score: {overall['mean_score']:.4f} ± {overall['std_score']:.4f}")

## 6. Fine-tune the Model (Optional)

**Note**: Fine-tuning can take a long time and requires significant computational resources. 
This section is optional and can be skipped if you just want to use the pre-trained model.

In [ ]:
# Uncomment the following code to run fine-tuning
# WARNING: This can take several hours depending on your hardware

# print("Starting model fine-tuning...")
# print("This may take several hours. Consider using a smaller dataset for testing.")

# # Fine-tune the model
# fine_tune_results = abstractive_summarizer.fine_tune_on_legal_data(
#     train_data=training_data[:100],  # Use smaller subset for demo
#     val_data=validation_data[:50],
#     output_dir='../models/legal_t5_finetuned',
#     num_epochs=2,  # Reduced for demo
#     batch_size=2,
#     learning_rate=3e-5
# )

# print("Fine-tuning completed!")
# print(f"Model saved to: {fine_tune_results['output_dir']}")

## 7. Test Extractive Summarization

In [ ]:
# Initialize extractive summarizer
extractive_summarizer = LegalExtractiveSummarizer()

print("Testing extractive summarization...")

extractive_results = []
for i, (doc, ref_summary) in enumerate(zip(test_documents, test_summaries)):
    print(f"\nProcessing sample {i+1}/{test_samples}")
    
    # Generate extractive summary
    result = extractive_summarizer.generate_extractive_summary(
        doc,
        num_sentences=5,
        method='textrank'
    )
    
    generated_summary = result['summary']
    
    extractive_results.append({
        'document': doc,
        'reference_summary': ref_summary,
        'generated_summary': generated_summary,
        'metadata': result
    })
    
    print(f"Reference length: {len(ref_summary.split())} words")
    print(f"Generated length: {len(generated_summary.split())} words")
    print(f"Selected sentences: {result['num_sentences']}")
    print(f"Generated summary: {generated_summary[:150]}...")

## 8. Evaluate Extractive Summarization

In [ ]:
# Evaluate extractive summarization
extractive_generated = [result['generated_summary'] for result in extractive_results]
extractive_reference = [result['reference_summary'] for result in extractive_results]
extractive_documents = [result['document'] for result in extractive_results]

extractive_evaluation = evaluator.evaluate_summaries(
    generated_summaries=extractive_generated,
    reference_summaries=extractive_reference,
    documents=extractive_documents
)

print("Extractive Summarization Evaluation Results:")
print("=" * 50)

extractive_aggregate = extractive_evaluation['aggregate_scores']

if 'rouge' in extractive_aggregate:
    rouge_scores = extractive_aggregate['rouge']
    print(f"ROUGE-1 F1: {rouge_scores['rouge1_fmeasure_mean']:.4f}")
    print(f"ROUGE-2 F1: {rouge_scores['rouge2_fmeasure_mean']:.4f}")
    print(f"ROUGE-L F1: {rouge_scores['rougeL_fmeasure_mean']:.4f}")

if 'overall' in extractive_aggregate:
    overall = extractive_aggregate['overall']
    print(f"Overall Score: {overall['mean_score']:.4f} ± {overall['std_score']:.4f}")

## 9. Test Hybrid Summarization

In [ ]:
# Initialize hybrid summarizer
hybrid_summarizer = LegalHybridSummarizer(
    extractive_summarizer=extractive_summarizer,
    abstractive_summarizer=abstractive_summarizer
)

print("Testing hybrid summarization...")

hybrid_results = []
for i, (doc, ref_summary) in enumerate(zip(test_documents, test_summaries)):
    print(f"\nProcessing sample {i+1}/{test_samples}")
    
    # Generate hybrid summary
    result = hybrid_summarizer.optimize_hybrid_output(
        doc,
        target_length=200
    )
    
    if 'error' not in result:
        generated_summary = result['summary']
        
        hybrid_results.append({
            'document': doc,
            'reference_summary': ref_summary,
            'generated_summary': generated_summary,
            'metadata': result
        })
        
        print(f"Reference length: {len(ref_summary.split())} words")
        print(f"Generated length: {len(generated_summary.split())} words")
        print(f"Strategy used: {result.get('strategy', 'unknown')}")
        print(f"Quality score: {result.get('quality_score', 0):.3f}")
        print(f"Generated summary: {generated_summary[:150]}...")
    else:
        print(f"Error in hybrid summarization: {result['error']}")

## 10. Compare All Methods

In [ ]:
# Collect results from all methods
method_results = {}

if pretrained_results:
    # Evaluate abstractive results
    abs_generated = [r['generated_summary'] for r in pretrained_results]
    abs_reference = [r['reference_summary'] for r in pretrained_results]
    abs_documents = [r['document'] for r in pretrained_results]
    
    abs_evaluation = evaluator.evaluate_summaries(
        generated_summaries=abs_generated,
        reference_summaries=abs_reference,
        documents=abs_documents
    )
    method_results['abstractive'] = abs_evaluation['individual_scores']

if extractive_results:
    # Evaluate extractive results
    ext_generated = [r['generated_summary'] for r in extractive_results]
    ext_reference = [r['reference_summary'] for r in extractive_results]
    ext_documents = [r['document'] for r in extractive_results]
    
    ext_evaluation = evaluator.evaluate_summaries(
        generated_summaries=ext_generated,
        reference_summaries=ext_reference,
        documents=ext_documents
    )
    method_results['extractive'] = ext_evaluation['individual_scores']

if hybrid_results:
    # Evaluate hybrid results
    hyb_generated = [r['generated_summary'] for r in hybrid_results]
    hyb_reference = [r['reference_summary'] for r in hybrid_results]
    hyb_documents = [r['document'] for r in hybrid_results]
    
    hyb_evaluation = evaluator.evaluate_summaries(
        generated_summaries=hyb_generated,
        reference_summaries=hyb_reference,
        documents=hyb_documents
    )
    method_results['hybrid'] = hyb_evaluation['individual_scores']

# Compare methods
if method_results:
    comparison = evaluator.compare_methods(method_results)
    
    print("Method Comparison:")
    print("=" * 40)
    
    for method, rank in comparison['ranking'].items():
        scores = comparison['method_scores'][method]
        print(f"{rank}. {method.title()}: {scores['mean']:.4f} ± {scores['std']:.4f}")
    
    # Create comparison plot
    plt.figure(figsize=(10, 6))
    
    methods = list(comparison['method_scores'].keys())
    means = [comparison['method_scores'][m]['mean'] for m in methods]
    stds = [comparison['method_scores'][m]['std'] for m in methods]
    
    bars = plt.bar(methods, means, yerr=stds, capsize=5, alpha=0.7)
    plt.title('Summarization Methods Comparison')
    plt.ylabel('Overall Score')
    plt.xlabel('Method')
    
    # Add value labels on bars
    for bar, mean in zip(bars, means):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_y() + 0.01,
                 f'{mean:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print("No results to compare. Please run the previous cells first.")

## 11. Save Results

In [ ]:
# Save evaluation results
results_to_save = {
    'pretrained_abstractive': pretrained_results if 'pretrained_results' in locals() else [],
    'extractive': extractive_results if 'extractive_results' in locals() else [],
    'hybrid': hybrid_results if 'hybrid_results' in locals() else [],
    'method_comparison': comparison if 'comparison' in locals() else {}
}

# Save to JSON file
with open('../data/processed_data/training_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_to_save, f, indent=2, ensure_ascii=False, default=str)

print("Training results saved to ../data/processed_data/training_results.json")

# Also save method comparison as CSV
if 'comparison' in locals() and comparison:
    comparison_df = pd.DataFrame([
        {
            'method': method,
            'mean_score': scores['mean'],
            'std_score': scores['std'],
            'min_score': scores['min'],
            'max_score': scores['max'],
            'rank': comparison['ranking'][method]
        }
        for method, scores in comparison['method_scores'].items()
    ])
    
    comparison_df.to_csv('../data/processed_data/method_comparison.csv', index=False)
    print("Method comparison saved to ../data/processed_data/method_comparison.csv")

## Summary

This notebook demonstrated:

1. **Data Loading and Preparation**: Loaded the IN-Abs dataset and created training/validation splits.

2. **Pre-trained Model Evaluation**: Tested the performance of pre-trained T5 model on legal documents.

3. **Extractive Summarization**: Implemented and evaluated graph-based extractive summarization.

4. **Hybrid Summarization**: Combined extractive and abstractive methods for optimal performance.

5. **Method Comparison**: Compared all approaches using domain-specific evaluation metrics.

The results show that different methods have different strengths:
- **Extractive**: High precision, good for preserving key legal information
- **Abstractive**: Better readability and fluency
- **Hybrid**: Balances precision and readability

The choice of method depends on the specific use case and requirements.